## Churn Data — Feature Engineering

### By:
jdg

### Date:
2026-03-03

### Description:

Builds a reusable scikit-learn preprocessing pipeline that transforms the
primary data (`churn_primary.parquet`) into model-ready features.

Key decisions from the variable analysis (Step 3):
- **Drop**: `gender`, `PhoneService`, `TotalCharges`
- **Consolidate**: "No internet/phone service" → "No"
- **Add**: `is_new_customer` binary (tenure ≤ 6 months)
- **Preserve** class imbalance (~26% churn) — no SMOTE

The pipeline is built **unfitted** here. In Step 5, it will be wrapped
with a model into a full `Pipeline` and used inside cross-validation.

## 📚 Import libraries

In [1]:
import sys
from pathlib import Path

import pandas as pd

# Add src to path so we can import project modules
sys.path.insert(0, str(Path("../../src").resolve()))

from data.transformation import build_feature_pipeline, load_feature_config

### Load configuration

In [2]:
config = load_feature_config(Path("../../conf/data_preparation/features.yml"))
config

{'target': 'Churn',
 'drop_columns': ['gender', 'PhoneService', 'TotalCharges'],
 'numeric_columns': ['tenure', 'MonthlyCharges'],
 'boolean_columns': ['SeniorCitizen',
  'Partner',
  'Dependents',
  'PaperlessBilling'],
 'consolidate_ohe_columns': ['MultipleLines',
  'OnlineSecurity',
  'OnlineBackup',
  'DeviceProtection',
  'TechSupport',
  'StreamingTV',
  'StreamingMovies'],
 'multi_ohe_columns': ['InternetService', 'PaymentMethod'],
 'contract_column': 'Contract',
 'contract_order': ['Month-to-month', 'One year', 'Two year'],
 'service_level_replacements': {'No internet service': 'No',
  'No phone service': 'No'},
 'new_customer_threshold': 6,
 'numeric_impute_strategy': 'median',
 'categorical_impute_strategy': 'most_frequent',
 'boolean_impute_strategy': 'most_frequent',
 'input_path': 'data/03_primary/Churn/churn_primary.parquet',
 'feature_output_path': 'data/04_feature/Churn'}

## 💾 Load data

In [3]:
PRIMARY_PATH = Path("../../") / config["input_path"]

df = pd.read_parquet(PRIMARY_PATH, dtype_backend="numpy_nullable")
print(f"Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()

Loaded: 7,194 rows x 20 columns


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,False,True,False,1,False,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,True,Electronic check,29.85,29.85,False
1,Male,False,False,False,34,True,No,DSL,Yes,No,Yes,No,No,No,One year,False,Mailed check,56.95,1889.5,False
2,Male,False,False,False,2,True,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,True,Mailed check,53.85,108.15,True
3,Male,False,False,False,45,False,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,False,Bank transfer (automatic),42.3,1840.75,False
4,Female,False,False,False,2,True,No,Fiber optic,No,No,No,No,No,No,Month-to-month,True,Electronic check,70.7,151.65,True


## 👷 Separate X and y

In [4]:
target = config["target"]

X = df.drop(columns=[target])
y = df[target].astype(int)

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print("\nClass distribution:")
print(y.value_counts())
print(f"\nChurn rate: {y.mean():.1%}")

X shape: (7194, 19)
y shape: (7194,)

Class distribution:
Churn
0    5285
1    1909
Name: count, dtype: int64

Churn rate: 26.5%


## 🔧 Build preprocessing pipeline

In [5]:
pipeline = build_feature_pipeline(config)
pipeline

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('boolean', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_n

## ⚙️ Fit & transform (full data, for inspection only)

In [6]:
X_transformed = pipeline.fit_transform(X)
print(f"Output shape: {X_transformed.shape}")
X_transformed.head()

Output shape: (7194, 22)


,tenure,MonthlyCharges,SeniorCitizen,Partner,Dependents,PaperlessBilling,MultipleLines_Yes,OnlineSecurity_Yes,OnlineBackup_Yes,DeviceProtection_Yes,...,StreamingMovies_Yes,InternetService_DSL,InternetService_Fiber optic,InternetService_No,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,Contract,is_new_customer
0,1.0,29.85,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
1,34.0,56.95,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
2,2.0,53.85,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
3,45.0,42.30,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
4,2.0,70.70,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0


### Verify output

In [7]:
print(f"Shape: {X_transformed.shape}")
print(f"NaN count: {X_transformed.isna().sum().sum()}")
print(f"\nDtypes:\n{X_transformed.dtypes.value_counts()}")
print(f"\nColumns:\n{list(X_transformed.columns)}")

Shape: (7194, 22)
NaN count: 0

Dtypes:
float64    22
Name: count, dtype: int64

Columns:
['tenure', 'MonthlyCharges', 'SeniorCitizen', 'Partner', 'Dependents', 'PaperlessBilling', 'MultipleLines_Yes', 'OnlineSecurity_Yes', 'OnlineBackup_Yes', 'DeviceProtection_Yes', 'TechSupport_Yes', 'StreamingTV_Yes', 'StreamingMovies_Yes', 'InternetService_DSL', 'InternetService_Fiber optic', 'InternetService_No', 'PaymentMethod_Bank transfer (automatic)', 'PaymentMethod_Credit card (automatic)', 'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check', 'Contract', 'is_new_customer']


In [8]:
X_transformed.describe()

,tenure,MonthlyCharges,SeniorCitizen,Partner,Dependents,PaperlessBilling,MultipleLines_Yes,OnlineSecurity_Yes,OnlineBackup_Yes,DeviceProtection_Yes,...,StreamingMovies_Yes,InternetService_DSL,InternetService_Fiber optic,InternetService_No,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,Contract,is_new_customer
count,7194.000000,7194.000000,7194.000000,7194.000000,7194.000000,7194.000000,7194.000000,7194.000000,7194.000000,7194.000000,...,7194.000000,7194.000000,7194.000000,7194.000000,7194.000000,7194.000000,7194.000000,7194.000000,7194.000000,7194.00000
mean,32.388935,64.838824,0.160967,0.481095,0.298026,0.595496,0.417014,0.282875,0.338338,0.338477,...,0.381846,0.340979,0.445649,0.213372,0.218237,0.214901,0.340562,0.226300,0.687517,0.21643
std,24.383657,29.940131,0.367526,0.499677,0.457423,0.490830,0.493100,0.450427,0.473177,0.473224,...,0.485873,0.474071,0.497072,0.409717,0.413078,0.410782,0.473931,0.418464,0.832866,0.41184
min,0.000000,18.250000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000
25%,9.000000,36.012500,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000
50%,29.000000,70.350000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000
75%,55.000000,89.850000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000,1.000000,0.000000,1.000000,0.00000
max,72.000000,114.850000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,2.000000,1.00000


### Inspect sub-pipelines

In [9]:
# Numeric imputer statistics
numeric_imputer = pipeline.named_transformers_["numeric"]["imputer"]
print("Numeric imputer statistics (median):")
for col, stat in zip(config["numeric_columns"], numeric_imputer.statistics_, strict=False):
    print(f"  {col}: {stat}")

# OHE categories for consolidate_ohe
consolidate_encoder = pipeline.named_transformers_["consolidate_ohe"]["encoder"]
print(f"\nConsolidate OHE categories: {consolidate_encoder.categories_}")

# OHE categories for multi_ohe
multi_encoder = pipeline.named_transformers_["multi_ohe"]["encoder"]
print(f"\nMulti OHE categories: {multi_encoder.categories_}")

# Ordinal encoding for contract
contract_encoder = pipeline.named_transformers_["contract"]["encoder"]
print(f"\nContract ordinal categories: {contract_encoder.categories_}")

Numeric imputer statistics (median):
  tenure: 29.0
  MonthlyCharges: 70.35

Consolidate OHE categories: [array(['No', 'Yes'], dtype=object), array(['No', 'Yes'], dtype=object), array(['No', 'Yes'], dtype=object), array(['No', 'Yes'], dtype=object), array(['No', 'Yes'], dtype=object), array(['No', 'Yes'], dtype=object), array(['No', 'Yes'], dtype=object)]

Multi OHE categories: [array(['DSL', 'Fiber optic', 'No'], dtype=object), array(['Bank transfer (automatic)', 'Credit card (automatic)',
       'Electronic check', 'Mailed check'], dtype=object)]

Contract ordinal categories: [array(['Month-to-month', 'One year', 'Two year'], dtype=object)]


### Verify is_new_customer

In [10]:
# Cross-tab is_new_customer with tenure ranges
tenure_bins = pd.cut(
    X["tenure"].astype(float),
    bins=[0, 6, 12, 24, 48, 72],
    labels=["0-6", "7-12", "13-24", "25-48", "49-72"],
    include_lowest=True,
)
cross = pd.crosstab(tenure_bins, X_transformed["is_new_customer"])
print("is_new_customer cross-tab with tenure ranges:")
cross

is_new_customer cross-tab with tenure ranges:


is_new_customer,0.0,1.0
tenure,,
0-6,0,1482
7-12,714,0
13-24,1041,0
25-48,1614,0
49-72,2268,0


## 💾 Save artifacts

In [11]:
output_path = Path("../../") / config["feature_output_path"]
output_path.mkdir(parents=True, exist_ok=True)

features_path = output_path / "features.parquet"
target_path = output_path / "target.parquet"

X_transformed.to_parquet(features_path, index=False)
y.to_frame().to_parquet(target_path, index=False)

print(f"Saved features: {features_path} ({X_transformed.shape})")
print(f"Saved target:   {target_path} ({y.shape[0]:,} rows)")

Saved features: ../../data/04_feature/Churn/features.parquet ((7194, 22))
Saved target:   ../../data/04_feature/Churn/target.parquet (7,194 rows)


## 📊 Analysis of Results and Conclusions

- **Input**: 7,194 rows × 19 features (after dropping target)
- **Output**: 7,194 rows × 22 features, all float64, zero NaN
- **Dropped**: `gender` (low predictive power), `PhoneService` (low predictive power),
  `TotalCharges` (multicollinear with tenure, r=0.82)
- **Consolidation**: "No internet/phone service" → "No" reduced 7 columns from 3 levels
  to 2 (binary after OHE with `drop='if_binary'`)
- **New feature**: `is_new_customer` captures the high-churn short-tenure segment
- **Imputation**: median for numeric, mode for boolean/categorical — all inside the
  pipeline to prevent data leakage during cross-validation
- The pipeline is built **unfitted** and importable from `src/data/transformation`
  for direct use in Step 5's model pipeline

## 💡 Proposals and Ideas

- Proceed to `5-models/`: wrap `build_feature_pipeline()` + model in a full
  `Pipeline`, evaluate with `cross_val_score` using `StratifiedKFold`
- Models to try: Logistic Regression (baseline), Random Forest, Gradient Boosting
- Use `class_weight='balanced'` to handle the ~26% churn imbalance
- Hyperparameter tuning via `GridSearchCV` or `RandomizedSearchCV`